In [ ]:
# %pip install sentencepiece protobuf tiktoken

# =====================================================
# 1. IMPORTACIONES
# =====================================================

import pandas as pd
import random
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

from torch.utils.data import Dataset
from sklearn.metrics import f1_score

In [ ]:
# =====================================================
# 2. CONFIGURACIÓN DE REPRODUCIBILIDAD
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("Semilla fijada:", SEED)

In [ ]:
# =====================================================
# 3. CARGA DE LOS CONJUNTOS DE DATOS
# =====================================================

# Carpeta donde están guardados los splits creados anteriormente
carpeta_splits = r"./Bases de Datos Splits/"

# -----------------------------------------------------
# TRAIN
# -----------------------------------------------------
# Para Train se utilizan los conjuntos balanceados,
# porque el balanceo solo se aplicó sobre los datos de entrenamiento.

train_DATD_bal = pd.read_csv(carpeta_splits + "train_DATD_balanceado.csv")
train_SDCNL_bal = pd.read_csv(carpeta_splits + "train_SDCNL_balanceado.csv")
train_DU_bal = pd.read_csv(carpeta_splits + "train_DU_balanceado.csv")

# -----------------------------------------------------
# VALIDATION Y TEST
# -----------------------------------------------------
# Validation y Test NO se balancean.
# Se mantienen los conjuntos originales para evaluar el modelo
# sobre una distribución de datos que no ha sido modificada.

val_DATD = pd.read_csv(carpeta_splits + "val_DATD.csv")
test_DATD = pd.read_csv(carpeta_splits + "test_DATD.csv")

val_SDCNL = pd.read_csv(carpeta_splits + "val_SDCNL.csv")
test_SDCNL = pd.read_csv(carpeta_splits + "test_SDCNL.csv")

val_DU = pd.read_csv(carpeta_splits + "val_DU.csv")
test_DU = pd.read_csv(carpeta_splits + "test_DU.csv")

print("Datasets cargados correctamente.")

In [ ]:
# =====================================================
# 4. SELECCIÓN DEL TRANSFORMER
# =====================================================

# Modelo 1: RoBERTa Base
# nombre_modelo = "roberta-base"

# Modelo 2: DeBERTa v3 Base
nombre_modelo = "microsoft/deberta-v3-base"

# Cargar el tokenizer correspondiente al Transformer seleccionado
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Longitud máxima indicada para los textos
MAX_LENGTH = 128

print("Modelo seleccionado:", nombre_modelo)
print("Longitud máxima:", MAX_LENGTH)

In [ ]:
# =====================================================
# 5. FUNCIÓN DE TOKENIZACIÓN, PADDING Y TRUNCATION
# =====================================================

def tokenizar(df):

    # Se pasa únicamente la columna Text al tokenizer.
    #
    # padding="max_length":
    # rellena los textos más cortos hasta llegar a 128 tokens.
    #
    # truncation=True:
    # recorta los textos que superen los 128 tokens.
    #
    # max_length=MAX_LENGTH:
    # fija la longitud final de todas las secuencias en 128.

    return tokenizer(
        df["Text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
# =====================================================
# 6. TOKENIZAR TRAIN, VALIDATION Y TEST
# =====================================================

# Cada Transformer necesita su propio tokenizer. Por eso esta tokenización se vuelve a ejecutar cuando se cambia de RoBERTa a DeBERTa.

# DATD
tokens_train_DATD = tokenizar(train_DATD_bal)
tokens_val_DATD = tokenizar(val_DATD)
tokens_test_DATD = tokenizar(test_DATD)

# SDCNL
tokens_train_SDCNL = tokenizar(train_SDCNL_bal)
tokens_val_SDCNL = tokenizar(val_SDCNL)
tokens_test_SDCNL = tokenizar(test_SDCNL)

# DU
tokens_train_DU = tokenizar(train_DU_bal)
tokens_val_DU = tokenizar(val_DU)
tokens_test_DU = tokenizar(test_DU)

print("Tokenización, padding y truncation finalizados correctamente.")

In [ ]:
# =====================================================
# 7. PREPARACIÓN DE LOS DATOS PARA EL TRAINER
# =====================================================

# Esta clase junta:
# - input_ids
# - attention_mask
# - labels
#
# De esta forma cada instancia queda preparada
# para ser utilizada por Hugging Face Trainer.

class DatasetTexto(Dataset):

    def __init__(self, tokens, labels):

        # Guardar los tokens obtenidos anteriormente
        self.tokens = tokens

        # Guardar las etiquetas correspondientes a cada texto
        self.labels = labels.tolist()

    def __len__(self):

        # Devolver el número total de instancias
        return len(self.labels)

    def __getitem__(self, idx):

        # Devolver una instancia concreta en formato tensor
        return {
            "input_ids": torch.tensor(self.tokens["input_ids"][idx]),
            "attention_mask": torch.tensor(self.tokens["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# =====================================================
# 8. CREACIÓN DE LOS DATASETS PARA EL TRAINER
# =====================================================

# DATD
dataset_train_DATD = DatasetTexto(tokens_train_DATD, train_DATD_bal["Label"])
dataset_val_DATD = DatasetTexto(tokens_val_DATD, val_DATD["Label"])
dataset_test_DATD = DatasetTexto(tokens_test_DATD, test_DATD["Label"])

# SDCNL
# Original: 1=depresión, 2=suicidio
# Para el modelo: 0=depresión, 1=suicidio
labels_train_SDCNL = train_SDCNL_bal["Label"].map({1: 0, 2: 1})
labels_val_SDCNL = val_SDCNL["Label"].map({1: 0, 2: 1})
labels_test_SDCNL = test_SDCNL["Label"].map({1: 0, 2: 1})

dataset_train_SDCNL = DatasetTexto(tokens_train_SDCNL, labels_train_SDCNL)
dataset_val_SDCNL = DatasetTexto(tokens_val_SDCNL, labels_val_SDCNL)
dataset_test_SDCNL = DatasetTexto(tokens_test_SDCNL, labels_test_SDCNL)

# DU
dataset_train_DU = DatasetTexto(tokens_train_DU, train_DU_bal["Label"])
dataset_val_DU = DatasetTexto(tokens_val_DU, val_DU["Label"])
dataset_test_DU = DatasetTexto(tokens_test_DU, test_DU["Label"])

print("Datasets preparados correctamente.")

In [ ]:
# =====================================================
# 9. MÉTRICAS
# =====================================================

# -----------------------------------------------------
# F1 PARA LOS MODELOS BINARIOS - DATD Y SDCNL
# -----------------------------------------------------
def compute_metrics_binario(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1 = f1_score(
        labels,
        predictions,
        average="binary",
        pos_label=1
    )

    return {"f1": f1}

# -----------------------------------------------------
# F1 PARA EL MODELO MULTICLASE - DU
# -----------------------------------------------------
def compute_metrics_multiclase(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {"f1": f1}

In [ ]:
# =====================================================
# 10. CONFIGURACIÓN DE LA BASELINE
# =====================================================

# Hiperparámetros fijos y comunes para establecer
# una línea base antes de utilizar los valores de Optuna.

BASELINE_BATCH_SIZE = 16
BASELINE_LEARNING_RATE = 2e-5
BASELINE_WEIGHT_DECAY = 0.01
BASELINE_EPOCHS = 10
BASELINE_PATIENCE = 3
BASELINE_WARMUP_STEPS = 100

print("Configuración baseline:")
print("Batch size:", BASELINE_BATCH_SIZE)
print("Learning rate:", BASELINE_LEARNING_RATE)
print("Weight decay:", BASELINE_WEIGHT_DECAY)
print("Épocas máximas:", BASELINE_EPOCHS)
print("Early stopping:", BASELINE_PATIENCE)
print("Warmup steps:", BASELINE_WARMUP_STEPS)

In [ ]:
# =====================================================
#  11. FUNCIÓN GENERAL PARA ENTRENAR UNA BASELINE
# =====================================================

import gc

def entrenar_baseline(nombre_tarea, num_labels, dataset_train, dataset_val, compute_metrics):

    # Reiniciar la semilla antes de crear cada modelo
    set_seed(SEED)

    # Crear el modelo en precisión float32 para mantener estabilidad numérica durante el entrenamiento
    model_baseline = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=num_labels,
        dtype=torch.float32
    )

    # Configuración del entrenamiento
    args_baseline = TrainingArguments(
        output_dir=f"./baseline/{nombre_modelo}/{nombre_tarea}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        # Hiperparámetros fijos de la baseline
        learning_rate=BASELINE_LEARNING_RATE,
        per_device_train_batch_size=BASELINE_BATCH_SIZE,
        per_device_eval_batch_size=BASELINE_BATCH_SIZE,
        weight_decay=BASELINE_WEIGHT_DECAY,
        num_train_epochs=BASELINE_EPOCHS,
        
        # Warmup para hacer más estable el inicio del entrenamiento
        warmup_steps=BASELINE_WARMUP_STEPS,
        
        # Limitar gradientes demasiado grandes
        max_grad_norm=1.0,

        seed=SEED,
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none"
    )

    # Crear el Trainer
    trainer_baseline = Trainer(
        model=model_baseline,
        args=args_baseline,
        train_dataset=dataset_train,
        eval_dataset=dataset_val,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=BASELINE_PATIENCE
            )
        ]
    )

    print("\n========================================")
    print("BASELINE:", nombre_tarea)
    print("Transformer:", nombre_modelo)
    print("========================================")

    # Entrenar
    trainer_baseline.train()

    # =====================================================
    # GUARDAR EL MEJOR RESULTADO DE LA BASELINE
    # =====================================================
    # Recuperar el mejor F1 obtenido sobre Validation
    mejor_f1 = trainer_baseline.state.best_metric
    # Mostrar claramente a qué modelo y dataset pertenece
    print("\n========================================")
    print("RESULTADO BASELINE")
    print("========================================")
    print("Modelo:", nombre_modelo)
    print("Dataset:", nombre_tarea)
    print("Mejor F1:", mejor_f1)
 
    #-------------------
    # Liberar el Trainer y el modelo antes de pasar
    # automáticamente al siguiente dataset
    del trainer_baseline
    del model_baseline

    # Liberar memoria de Python y de la GPU
    gc.collect()
    torch.cuda.empty_cache()

    return mejor_f1

In [ ]:
# =====================================================
# 12. EJECUTAR TODAS LAS BASELINES
# =====================================================

# DATD
f1_DATD_base = entrenar_baseline(
    "DATD",
    2,
    dataset_train_DATD,
    dataset_val_DATD,
    compute_metrics_binario
)

# SDCNL
f1_SDCNL_base = entrenar_baseline(
    "SDCNL",
    2,
    dataset_train_SDCNL,
    dataset_val_SDCNL,
    compute_metrics_binario
)

# DU
f1_DU_base = entrenar_baseline(
    "DU",
    3,
    dataset_train_DU,
    dataset_val_DU,
    compute_metrics_multiclase
)

print("\n========================================")
print("TODAS LAS BASELINES HAN FINALIZADO")
print("========================================")

In [ ]:
# =====================================================
# 13. RESUMEN DE RESULTADOS DE LAS BASELINES
# =====================================================

resultados_baseline = pd.DataFrame({
    "Modelo": [nombre_modelo, nombre_modelo, nombre_modelo],
    "Dataset": ["DATD", "SDCNL", "DU"],
    "F1_Baseline": [
        f1_DATD_base,
        f1_SDCNL_base,
        f1_DU_base
    ]
})

print("\n========== RESULTADOS BASELINE ==========")
print(resultados_baseline)

In [ ]:
# =====================================================
# 14. GUARDAR RESULTADOS BASELINE
# =====================================================

# Carpeta donde se guardarán los resultados
carpeta_resultados = r"./Resultados Entrenamiento/"

# Crear un nombre corto según el Transformer seleccionado
if nombre_modelo == "roberta-base":
    nombre_archivo = "RoBERTa"
    
elif nombre_modelo == "microsoft/deberta-v3-base":
    nombre_archivo = "DeBERTa"

# Guardar automáticamente con el nombre del modelo correspondiente
resultados_baseline.to_csv(
    carpeta_resultados + f"baseline_{nombre_archivo}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Resultados baseline de {nombre_archivo} guardados correctamente.")